# Lab 1 — Geometry of Linear Transformations with Word Embeddings

**Linear Algebra · Upskilled**

In this lab you will connect the abstract machinery of norms, dot products, and linear maps to the concrete geometry of GloVe word embeddings. By the end you will have:

1. Measured how GloVe encodes meaning as *direction* rather than *magnitude*.
2. Built batch cosine-similarity in both **PyTorch** and **TensorFlow** using matrix multiplication.
3. Visualised how 2 × 2 matrices deform the unit circle — rotation, shear, reflection, projection.
4. Implemented orthogonal projection and the General = Particular + Homogeneous decomposition.
5. Found the gender subspace of GloVe and projected it out.

---
> **Colab tip**: Runtime → Run all  (Ctrl+F9) executes every cell. GPU is not required.

## 0 · Setup

We download the 50-dimensional GloVe vectors (822 MB unzipped; ~170 MB download). The first run takes ~2 minutes; subsequent runs use a cached copy.

In [ ]:
# ── dependencies ────────────────────────────────────────────────────────────
import subprocess, sys
def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

pip('torch')
pip('tensorflow')
pip('numpy')
pip('matplotlib')
pip('requests')

import numpy as np
import torch
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests, zipfile, io, os

print('torch   :', torch.__version__)
print('tf      :', tf.__version__)
print('numpy   :', np.__version__)

In [ ]:
# ── download GloVe 50-d ──────────────────────────────────────────────────────
GLOVE_PATH = 'glove.6B.50d.txt'

if not os.path.exists(GLOVE_PATH):
    print('Downloading GloVe 6B 50d …')
    url = 'https://nlp.stanford.edu/data/glove.6B.zip'
    r = requests.get(url, stream=True)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extract('glove.6B.50d.txt', '.')
    print('Done.')
else:
    print('Using cached GloVe file.')

# ── parse into dict and numpy array ─────────────────────────────────────────
words, vecs = [], []
with open(GLOVE_PATH, encoding='utf-8') as f:
    for line in f:
        parts = line.rstrip().split(' ')
        words.append(parts[0])
        vecs.append(list(map(float, parts[1:])))

word2idx = {w: i for i, w in enumerate(words)}
E = np.array(vecs, dtype=np.float32)   # shape: (400000, 50)

print(f'Vocabulary : {len(words):,} words')
print(f'Embedding matrix shape: {E.shape}')

---
## 1 · Norms and the Meaning of Magnitude

GloVe vectors are trained to encode *co-occurrence statistics*. Before comparing two vectors it is important to understand whether their raw magnitudes carry semantic information or are noise.

In [ ]:
# ── 1a. Compute L2 norms of every embedding ──────────────────────────────────
norms_np = np.linalg.norm(E, axis=1)          # shape: (400000,)

# Same computation in PyTorch
E_torch = torch.from_numpy(E)
norms_torch = torch.linalg.norm(E_torch, dim=1)

# Same in TensorFlow
E_tf = tf.constant(E)
norms_tf = tf.norm(E_tf, axis=1)

print('NumPy  mean norm :', norms_np.mean().round(4))
print('PyTorch mean norm:', norms_torch.mean().item().__round__(4))
print('TF     mean norm :', float(tf.reduce_mean(norms_tf).numpy()).__round__(4))

In [ ]:
# ── 1b. Plot norm distribution ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(norms_np, bins=100, color='#2a5757', edgecolor='none', alpha=0.85)
ax.axvline(norms_np.mean(), color='#63aac9', lw=2, label=f'mean = {norms_np.mean():.2f}')
ax.set_xlabel('L2 norm')
ax.set_ylabel('count')
ax.set_title('Distribution of GloVe 50-d embedding norms')
ax.legend()
plt.tight_layout()
plt.show()

# Inspect two specific words
for w in ['king', 'the', 'quantum', 'cat']:
    idx = word2idx[w]
    print(f'  ||{w:10s}|| = {norms_np[idx]:.4f}')

**Reflection 1a**: Do high-frequency words ("the", "a") have systematically larger or smaller norms than rare words? What does this suggest about using raw dot products $\vec{u}\cdot\vec{v}$ to measure similarity?

In [ ]:
# ── 1c. Exercise: L1 and L-inf norms ─────────────────────────────────────────
# TODO: compute the L1 norm of 'dog' and the L-inf norm of 'cat'
# using numpy, then verify with torch.

dog = E[word2idx['dog']]
cat = E[word2idx['cat']]

# Your code here
l1_dog  = None   # replace with np.linalg.norm(dog, ord=1)
linf_cat = None  # replace with np.linalg.norm(cat, ord=np.inf)

print(f'||dog||_1   = {l1_dog}')
print(f'||cat||_inf = {linf_cat}')

---
## 2 · Cosine Similarity: Direction as Meaning

Cosine similarity removes magnitude from the comparison:
$$\text{sim}(\vec{u},\vec{v}) = \frac{\vec{u}\cdot\vec{v}}{\|\vec{u}\|\,\|\vec{v}\|} = \hat{u}\cdot\hat{v}$$

Normalising to unit vectors first is the standard trick: once you have unit vectors, similarity is just a dot product — and batches of dot products are matrix multiplication.

In [ ]:
# ── 2a. Normalise the full embedding matrix ───────────────────────────────────
# NumPy
norms_col = norms_np[:, np.newaxis]          # (400000, 1) for broadcasting
E_unit = E / norms_col                       # unit-norm embeddings

# PyTorch — identical indexing semantics
E_unit_torch = E_torch / norms_torch.unsqueeze(1)

# TensorFlow
E_unit_tf = tf.math.l2_normalize(E_tf, axis=1)

# Sanity check: all norms should be 1
check = np.linalg.norm(E_unit, axis=1)
print(f'min norm after normalisation: {check.min():.6f}')
print(f'max norm after normalisation: {check.max():.6f}')

In [ ]:
# ── 2b. Batch cosine similarity via matrix multiplication ─────────────────────
# Pick a probe set of words
probe_words = ['king', 'queen', 'man', 'woman', 'dog', 'cat', 'paris', 'france']
probe_idx   = [word2idx[w] for w in probe_words]

# Extract unit-norm probe matrix  P: (8, 50)
P = E_unit[probe_idx]   # numpy

# Cosine similarity matrix: P @ P^T  gives (8, 8) — entry (i,j) = cos(theta_ij)
cos_mat = P @ P.T

# ── plot heatmap ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cos_mat, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(probe_words))); ax.set_xticklabels(probe_words, rotation=45, ha='right')
ax.set_yticks(range(len(probe_words))); ax.set_yticklabels(probe_words)
plt.colorbar(im, ax=ax, label='cosine similarity')
ax.set_title('Pairwise cosine similarity (GloVe 50-d)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2c. Same in PyTorch ───────────────────────────────────────────────────────
P_torch = E_unit_torch[probe_idx]
cos_mat_torch = P_torch @ P_torch.T    # matrix multiplication

# Verify numerically identical to numpy result
diff = (cos_mat_torch.numpy() - cos_mat).max()
print(f'Max abs diff (PyTorch vs NumPy): {diff:.2e}')  # should be ~1e-7

# ── 2d. Same in TensorFlow ───────────────────────────────────────────────────
P_tf = tf.gather(E_unit_tf, probe_idx)
cos_mat_tf = P_tf @ tf.transpose(P_tf)
print(f'Max abs diff (TF    vs NumPy): {float(tf.reduce_max(tf.abs(cos_mat_tf - cos_mat)).numpy()):.2e}')

In [ ]:
# ── 2e. Word analogy: king - man + woman ≈ queen ─────────────────────────────
def nearest(query_vec, top_k=5, exclude=()):
    """Return top_k most similar words by cosine similarity."""
    q = query_vec / np.linalg.norm(query_vec)
    scores = E_unit @ q                        # (400000,)
    order  = np.argsort(-scores)
    results = []
    for idx in order:
        if words[idx] not in exclude:
            results.append((words[idx], float(scores[idx])))
        if len(results) == top_k:
            break
    return results

analogy = E[word2idx['king']] - E[word2idx['man']] + E[word2idx['woman']]
print('king - man + woman:')
for w, s in nearest(analogy, exclude={'king', 'man', 'woman'}):
    print(f'  {w:15s}  {s:.4f}')

**Reflection 2a**: The analogy `king - man + woman` exploits the *linearity* of the embedding space. In your own words, why does vector subtraction remove the "maleness" direction and addition insert the "femaleness" direction?

---
## 3 · Visualising Linear Transformations

A 2 × 2 matrix $A$ defines a linear map $T_A(\vec{x}) = A\vec{x}$. Applying $T_A$ to every point on the unit circle reveals the transformation's geometry: how it stretches, rotates, or collapses the plane.

In [ ]:
# ── helpers ───────────────────────────────────────────────────────────────────
theta_vals = np.linspace(0, 2 * np.pi, 500)
unit_circle = np.stack([np.cos(theta_vals), np.sin(theta_vals)])  # (2, 500)

def plot_transform(A, title, ax):
    """Plot unit circle and its image under A on ax."""
    image = A @ unit_circle    # (2, 500)
    ax.plot(*unit_circle, color='#63aac9', lw=1.5, label='unit circle')
    ax.plot(*image,       color='#2a5757', lw=2.0, label='image')
    # draw axes
    lim = max(2, np.abs(image).max() * 1.2)
    ax.axhline(0, color='grey', lw=0.5); ax.axvline(0, color='grey', lw=0.5)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7)

# ── four canonical transforms ─────────────────────────────────────────────────
phi = np.pi / 4   # 45 degrees

transforms = [
    (np.array([[np.cos(phi), -np.sin(phi)],
               [np.sin(phi),  np.cos(phi)]]),  'Rotation by 45°'),
    (np.array([[1., 1.5],
               [0., 1.]]),                      'Horizontal shear'),
    (np.array([[2., 0.],
               [0., 0.5]]),                     'Anisotropic scaling'),
    (np.array([[1., 0.],
               [0., 0.]]),                      'Projection onto x-axis\n(rank-1, nullity=1)'),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for (A, title), ax in zip(transforms, axes):
    plot_transform(A, title, ax)
plt.suptitle('Images of the unit circle under linear maps', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3a. Exercise: determinant and area ────────────────────────────────────────
# The determinant of A is the signed area scale factor:
#   area(image of unit disk) = |det(A)| * area(unit disk) = |det(A)| * pi

for A, title in transforms:
    d = np.linalg.det(A)
    print(f'{title.split(chr(10))[0]:30s}  det = {d:+.4f}  |det| = {abs(d):.4f}')

In [ ]:
# ── 3b. Singular value decomposition reveals geometry ────────────────────────
# Every matrix A = U S V^T.  The singular values s_i are the semi-axes of
# the ellipse that is the image of the unit ball.

print(f'{"Transform":30s}  singular values')
print('-' * 55)
for A, title in transforms:
    _, S, _ = np.linalg.svd(A)
    print(f'{title.split(chr(10))[0]:30s}  {np.round(S, 4)}')

**Reflection 3a**: The projection onto the x-axis has singular values [1, 0]. Why does a zero singular value always imply rank deficiency? How does that connect to the collapsed circle you see in the plot?

---
## 4 · Orthogonal Projection and the Null Space

Given an orthonormal basis $\{\vec{q}_1, \ldots, \vec{q}_k\}$ for a subspace $W$, the orthogonal projection of $\vec{v}$ onto $W$ is:
$$\text{proj}_W(\vec{v}) = Q Q^T \vec{v} \qquad (Q = [\vec{q}_1 \mid \cdots \mid \vec{q}_k])$$

In [ ]:
# ── 4a. Build an orthonormal basis for a 2D subspace of R^50 ─────────────────
# Use Gram-Schmidt on two GloVe vectors
v1 = E[word2idx['man']] - E[word2idx['woman']]   # 'gender' direction
v2 = E[word2idx['king']] - E[word2idx['queen']]  # another gender-ish direction

q1 = v1 / np.linalg.norm(v1)
v2_orth = v2 - np.dot(q1, v2) * q1              # remove q1 component
q2 = v2_orth / np.linalg.norm(v2_orth)

Q = np.stack([q1, q2], axis=1)                   # shape: (50, 2)
print('Q shape:', Q.shape)
print('Q^T Q (should be I_2):')
print(np.round(Q.T @ Q, 6))

In [ ]:
# ── 4b. Project 'doctor' onto and off the gender subspace ────────────────────
v_doc = E[word2idx['doctor']]

proj_gender  = Q @ (Q.T @ v_doc)    # component in W
proj_orth    = v_doc - proj_gender  # component perpendicular to W

print(f'||doctor||            = {np.linalg.norm(v_doc):.4f}')
print(f'||proj onto gender||  = {np.linalg.norm(proj_gender):.4f}')
print(f'||proj off gender ||  = {np.linalg.norm(proj_orth):.4f}')
print(f'Pythagoras check      : {(np.linalg.norm(proj_gender)**2 + np.linalg.norm(proj_orth)**2):.6f}')
print(f'  vs ||doctor||^2     : {np.linalg.norm(v_doc)**2:.6f}')

In [ ]:
# ── 4c. Same projection in PyTorch ───────────────────────────────────────────
Q_torch   = torch.from_numpy(Q.astype(np.float32))
v_doc_t   = torch.from_numpy(v_doc.astype(np.float32))

proj_t    = Q_torch @ (Q_torch.T @ v_doc_t)
orth_t    = v_doc_t - proj_t

print('PyTorch ||proj onto gender||:', torch.linalg.norm(proj_t).item().__round__(4))

# ── 4d. Same in TensorFlow ───────────────────────────────────────────────────
Q_tf      = tf.constant(Q.astype(np.float32))
v_doc_tf  = tf.constant(v_doc.astype(np.float32))

proj_tf   = Q_tf @ tf.linalg.matvec(tf.transpose(Q_tf), v_doc_tf)[:, tf.newaxis]
proj_tf   = tf.squeeze(proj_tf)
print('TF     ||proj onto gender||:', float(tf.norm(proj_tf).numpy()).__round__(4))

In [ ]:
# ── 4e. Debiased embeddings: remove gender subspace from all words ────────────
P_proj = Q @ Q.T                                  # (50, 50) projection matrix
E_debiased = E - (E @ Q) @ Q.T                    # (400000, 50)

# Check: the 'he'/'she' analogy should now be weaker
def cos_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

pairs = [('he', 'she'), ('king', 'queen'), ('doctor', 'nurse')]
print(f'{"pair":20s}  original   debiased')
for w1, w2 in pairs:
    orig = cos_sim(E[word2idx[w1]],        E[word2idx[w2]])
    deb  = cos_sim(E_debiased[word2idx[w1]], E_debiased[word2idx[w2]])
    print(f'{w1:10s}/{w2:10s}  {orig:+.4f}     {deb:+.4f}')

**Reflection 4a**: The debiased embeddings are the *perpendicular complement* of the gender subspace. Does similarity between `king` and `queen` drop significantly? Why does projection-based debiasing not fully remove gender associations in practice?

---
## 5 · General = Particular + Homogeneous

The solution set of $A\vec{x} = \vec{b}$ has the form $\vec{x} = \vec{x}_p + \vec{x}_h$ where $\vec{x}_p$ is any particular solution and $\vec{x}_h$ ranges over the null space $\mathcal{N}(A)$. Here we make this concrete with a fat matrix (more unknowns than equations).

In [ ]:
# ── 5a. Set up an underdetermined system ─────────────────────────────────────
rng = np.random.default_rng(42)
A = rng.standard_normal((3, 6)).astype(np.float32)   # 3 equations, 6 unknowns
x_true = rng.standard_normal(6).astype(np.float32)
b = A @ x_true

# Particular solution via least-norm (pseudoinverse)
x_p, *_ = np.linalg.lstsq(A, b, rcond=None)
print(f'||A x_p - b|| = {np.linalg.norm(A @ x_p - b):.2e}  (should be ~0)')

# Null space via SVD
_, S, Vh = np.linalg.svd(A, full_matrices=True)
rank    = int(np.sum(S > 1e-6))
nullity = A.shape[1] - rank
null_basis = Vh[rank:]     # rows are null-space basis vectors, shape: (3, 6)
print(f'rank = {rank},  nullity = {nullity},  rank+nullity = {rank+nullity} = n = {A.shape[1]}')

In [ ]:
# ── 5b. Verify the decomposition ─────────────────────────────────────────────
# Any vector in the null space, e.g., first null basis vector
x_h = null_basis[0]

print(f'A @ x_h = {np.round(A @ x_h, 6)}  (should be 0)')

# x_p + x_h is also a solution
x_combo = x_p + 2.7 * null_basis[0] - 1.3 * null_basis[1]
print(f'||A(x_p + x_h) - b|| = {np.linalg.norm(A @ x_combo - b):.2e}')

# Visualise: the solution set is an affine subspace (coset of the null space)
fig, ax = plt.subplots(figsize=(7, 3))
solutions = [x_p + t * null_basis[0] for t in np.linspace(-3, 3, 200)]
norms_sol = [np.linalg.norm(s) for s in solutions]
ax.plot(np.linspace(-3, 3, 200), norms_sol, color='#2a5757', lw=2)
ax.axvline(0, color='#63aac9', ls='--', label='particular solution')
ax.set_xlabel('coefficient of null-space direction')
ax.set_ylabel('||solution||')
ax.set_title('Solution norm varies along the null space')
ax.legend()
plt.tight_layout()
plt.show()

**Reflection 5a**: The minimum-norm solution $\vec{x}_p$ is perpendicular to the null space. The plot shows that moving along the null-space direction *increases* the norm. Why does this make $\ell^2$ regularisation prefer the minimum-norm solution when solving underdetermined systems?

---
## 6 · Putting It Together: A Mini Semantic Search Engine

Semantic search over the GloVe vocabulary using only the machinery from this lab: normalisation, matrix multiplication, and nearest-neighbour lookup.

In [ ]:
# ── Build a semantic search function using batched cosine similarity ──────────

def semantic_search(query, top_k=10, exclude=None):
    """Return top_k nearest GloVe words to a query string or vector."""
    exclude = set(exclude or [])
    if isinstance(query, str):
        if query not in word2idx:
            print(f"'{query}' not in vocabulary")
            return []
        q_vec = E_unit[word2idx[query]]
        exclude.add(query)
    else:
        q_vec = query / np.linalg.norm(query)

    # Batch dot product: E_unit @ q_vec  (400000,)
    scores = E_unit @ q_vec
    order  = np.argsort(-scores)

    results = []
    for idx in order:
        if words[idx] not in exclude:
            results.append((words[idx], float(scores[idx])))
        if len(results) == top_k:
            break
    return results

# Test queries
for query in ['algebra', 'gradient', 'neural']:
    print(f'\n--- Nearest to "{query}" ---')
    for w, s in semantic_search(query, top_k=5):
        print(f'  {w:20s}  {s:.4f}')

In [ ]:
# ── Exercise: compose an analogy query using vector arithmetic ────────────────
# TODO: fill in the blanks to answer "paris is to france as berlin is to ___"

analogy_vec = (
    E[word2idx['paris']]
    - E[word2idx['france']]
    + E[word2idx['berlin']]
    # Hint: this should give 'germany'
)

print('paris - france + berlin ≈')
for w, s in semantic_search(analogy_vec, top_k=5, exclude={'paris', 'france', 'berlin'}):
    print(f'  {w:20s}  {s:.4f}')

---
## Summary

| Concept | What you did |
|---|---|
| **Norms** | Measured GloVe magnitudes; saw that raw magnitude encodes frequency, not meaning |
| **Cosine similarity** | Normalised to unit sphere; batched similarity = matrix multiplication |
| **Linear maps** | Visualised how det and singular values describe area scaling and collapse |
| **Orthogonal projection** | Built gender subspace via Gram-Schmidt; projected embeddings onto and off it |
| **Null space / G=P+H** | Confirmed rank-nullity numerically; saw that moving along null space preserves solutions |

**Next**: Lab 2 goes deeper into SVD and PCA, using the full GloVe matrix to explore how low-rank approximation and principal components relate to representation learning in neural networks.